In [1]:
from pybm.convert.probmot import load_library, load_model
import os 


library = load_library("./probmot/AquaticEcosystem.pbl")
incomplete_model = load_model("./probmot/BledIncompleteStructure.pbm", library)

In [2]:
import pandas as pd
from pybm.model import TimeSeries

names = ["00.data", "01.data", "02.data", "95-01.data", "95.data", "96.data", "97.data", "98.data", "99.data"]
dfs = []

# for name in names:
#     df = pd.read_csv("./data/" + name, sep=" ")
#     dfs.append(df)
# df  = pd.concat(dfs, ignore_index=True)

df = pd.read_csv("./data/00.data", sep=" ")

VAR_TO_COLUMN = {
    # eksogene (podatki, ki poganjajo model)
    "ortp.conc": "ortp",              # ortofosfat
    "no.conc": "no",                  # nitrat/dušik
    "silica.conc": "silica",          # silicij
    "daph.conc": "daph_lit",          # dafnija - v teh primerih EKSOGENA (opazovana), ne modelirana
    "env.temperature": "temp",
    "env.light": "light_m",           # NE "light" - uporabljajo "light_m" različico

    # endogena (tarča fita/simulacije)
    "phyto.conc": "phyto",
}

t_full = df["t"].values
for varname, column in VAR_TO_COLUMN.items():
    time_series = TimeSeries(t_full, df[column].values)
    incomplete_model.vars[varname].data = time_series

In [ ]:
t = t_full[0:100]  # only first 100 time points for estimation
# collocation times - subsample of the time points to use for gradient matching
collocation_times = t[::10]  # every 10th time point

In [ ]:
# get best model 
from pybm.estimate.estimate import estimate_model
results = estimate_model(incomplete_model, t_eval=t, recepie="gp", verbose=1, max_iter_loss=10, max_gp_iter=200, 
                         ftol=1e-3, xtol=1e-3, gtol=1e-3, max_nfev=200, collocation_times=collocation_times)

Structure estimation:   0%|          | 13/5184 [00:45<4:59:25,  3.47s/it]


KeyboardInterrupt: 

In [ ]:
# --- true vs. predicted: BledIncompleteParameters (ista struktura kot BledComplete, prave meritve) ---
# NE uporablja zgornjega `model` (BledIncompleteStructure) - tam so imena konstant odvisna od tega,
# katera strukturna izbira je bila (arbitrarno) izbrana v models[0], zato se ne ujemajo z BledComplete.
import numpy as np
from pybm.convert.probmot import load_model
from pybm.estimate.gradient_matching import estimate_gradient_matching

# "prave" vrednosti - prebrane direktno iz BledComplete.pbm (ProBMoT-ova že rešena vrednost za
# isto strukturo, kjer so tile isti parametri v BledIncompleteParameters.pbm postavljeni na null)
TRUE_VALUES = {
    "ortp.halfSaturation": 0.5604633993464766,
    "no.halfSaturation": 0.005439851527049321,
    "silica.halfSaturation": 4.239434744795273,
    "phyto.maxGrowthRate": 3.0,
    "daph.maxFiltrationRate": 0.03737605860906343,
    "lightInfluence19.optLight": 100.49292986819768,
    "tempGrowthLim906.refTemp": 10.0,
    "respirationPP381.respRate": 1.0e-4,
    "tempRespInfluence396.refTemp": 15.60799148612582,
}
TRUE_INITIAL_PHYTO_CONC = 2.3002014225415075

params_model = load_model("./probmot/BledIncompleteParameters.pbm", library)
[params_model] = params_model.induce()  # ni strukturne dvoumnosti - natanko en rezultat

for varname, column in VAR_TO_COLUMN.items():
    params_model.vars[varname].data = TimeSeries(t, df[column].values)
params_model.switch_engine("torch")

fit = estimate_gradient_matching(params_model, t, verbose=False)

print(f"{'spremenljivka':<32}{'napoved':>16}{'prava vrednost':>18}")
for name, true_value in TRUE_VALUES.items():
    try:
        predicted = fit.const_by_name[name]
        print(f"{name:<32}{predicted:>16.6g}{true_value:>18.6g}")
    except KeyError:
        print(f"{name:<32}{'N/A':>16}{'N/A':>18}")

# phyto.conc nima "initial_value" kot konstante - gradient matching ga ne fita direktno, ampak ga
# lahko preberemo iz GP-jevega glajenega posteriorja pri najzgodnejšem času kot najboljšo oceno
phyto_initial_pred = float(fit.gps["phyto.conc"].mean(np.array([t.min()]))[0])
print(f"{'phyto.conc (initial)':<32}{phyto_initial_pred:>16.6g}{TRUE_INITIAL_PHYTO_CONC:>18.6g}")
